# Phi-3.5 mini：未使用説明文72問で確認

T4 GPUが必要です。全セルを実行すると、新規出力先で1回評価して検算し、結果ZIPをダウンロードします。説明文60問で54正答以上・各概念8/10以上・none対照10/12以上。内容と回答形式を分けて採点します。結果を見て規則は変更しません。PLAN.mdを参照。

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install transformers==4.57.6 accelerate scikit-learn scipy pandas numpy sentencepiece


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, json, torch
assert torch.cuda.is_available(), 'GPUが利用できません。「ランタイム」→「ランタイムのタイプを変更」→T4 GPUにしてください。利用制限の場合はここで停止してください。'
print('GPU:', torch.cuda.get_device_name(0))
REPO=Path('/content/GraduationThesis-phi-validation')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ryouy/GraduationThesis.git',str(REPO)],check=True)
else:
    subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
FOLDER=REPO/'experiments/2026-11/1104_phi_validation'
import hashlib
frozen=json.loads((FOLDER/'FROZEN_SHA256.json').read_text())
for name, digest in frozen.items():
    assert hashlib.sha256((FOLDER/name).read_bytes()).hexdigest()==digest, name
subprocess.run([sys.executable,str(FOLDER/'check.py')],cwd=REPO,check=True)
OUT=Path('/content')/('phi-validation-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
print('OUTPUT:',OUT)
subprocess.run([sys.executable,'-u',str(FOLDER/'run.py'),'--resolve','--out',str(OUT)],cwd=REPO,check=True)
subprocess.run([sys.executable,'-u',str(FOLDER/'run.py'),'--key','phi3_5mini','--out',str(OUT)],cwd=REPO,check=True)


In [ ]:
import shutil
from google.colab import files
result=OUT/'phi3_5mini'
subprocess.run([sys.executable,str(FOLDER/'check.py'),str(result)],cwd=REPO,check=True)
manifest=json.loads((result/'manifest.json').read_text())
print(json.dumps({k:manifest[k] for k in ['status','correct','clear_n','min_per_concept','none_correct','format_correct','all_pass','truncations']},indent=2))
print('内容ゲート通過でも、原回答を点検するまで採用は確定しません。')
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
files.download(archive)
